# The classic NLP pipeline — a sentiment classifier

_Capstones_

**Walk the path that took natural-language processing from bag-of-words to attention: build a sentiment classifier out of word embeddings, a bidirectional LSTM, and an attention-pooling head.**

---

This notebook builds the pipeline one stage at a time — toy data, tokenizer, padded batches, word embeddings, a bidirectional LSTM, and an attention-pooling head that shows *where* the model read the sentiment.

The heart of the notebook is **attention pooling built tensor by tensor** on one concrete sentence: we watch token ids become embeddings, BiLSTM context vectors, raw attention scores, masked scores, softmax weights, one sentence vector, and final logits — then package the same steps into a reusable model.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders, dropdowns, and text boxes; outside Colab they run a sensible no-widget fallback. Run each cell top to bottom, experiment with the knobs, then tackle the practice ideas at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

Before any model, let's inspect the toy task. Each sentence is mostly neutral filler with one or two injected positive or negative words. Because both classes use the same length range and the same filler pool, a good classifier must learn **which words carry sentiment**, not a shortcut like sentence length.

### Step 1 — Generate the toy sentiment dataset

We keep the original seeds and data generator. The dataset is small on purpose: 240 training sentences and 60 test sentences are enough for the model to learn, while still letting us print, plot, and reason about every tensor.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Preserve the original lesson seeds.
torch.manual_seed(1)
random.seed(1)

# Word pools: positive, negative, and neutral filler.
pos_words = ["great","love","excellent","amazing","wonderful","fantastic",
             "brilliant","superb","delightful","gripping"]
neg_words = ["terrible","awful","horrible","boring","dreadful","disappointing",
             "tedious","lousy","painful","forgettable"]
neutral   = ["the","a","this","that","movie","film","story","plot","scene","cast",
             "i","we","it","was","is","really","saw","watched","with","and","of",
             "to","on","very","quite","about","some","one","first","last"]

def make(label, n):
    out = []
    for _ in range(n):
        L = random.randint(7, 12)                          # random sentence length
        words = [random.choice(neutral) for _ in range(L)] # start with all filler
        pool = pos_words if label == 1 else neg_words
        for _ in range(random.randint(1, 2)):              # inject 1-2 sentiment words
            words[random.randrange(L)] = random.choice(pool)
        out.append((" ".join(words), label))
    return out

train = make(1, 120) + make(0, 120)
test  = make(1, 30)  + make(0, 30)
random.shuffle(train)

print("train examples:", len(train), "  test examples:", len(test))
print("first training example:", train[0])

### Step 2 — A table of real examples

A model never sees our verbal description; it sees rows like these. The table includes the raw sentence, the label, its length, and which injected sentiment words appear.

In [ ]:
def tok(s):
    return s.split()

def sentiment_hits(s):
    words = tok(s)
    hits = [w for w in words if w in pos_words or w in neg_words]
    return ", ".join(hits)

sample_df = pd.DataFrame([
    {"sentence": s, "label": "positive" if y == 1 else "negative",
     "length": len(tok(s)), "sentiment word(s)": sentiment_hits(s)}
    for s, y in train[:10]
])
display(sample_df)

### Step 3 — Class balance and sentence lengths

The generator creates exactly balanced train/test splits. Lengths are deliberately similar across labels, so length alone should not solve the problem.

In [ ]:
all_rows = pd.DataFrame([
    {"split": "train", "label": y, "length": len(tok(s))} for s, y in train
] + [
    {"split": "test", "label": y, "length": len(tok(s))} for s, y in test
])
all_rows["label name"] = all_rows["label"].map({0: "negative", 1: "positive"})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
balance = all_rows.groupby(["split", "label name"]).size().unstack()
balance.plot(kind="bar", ax=ax1, color=["#ff6b6b", "#7ee787"])
ax1.set_title("class balance")
ax1.set_ylabel("count")
ax1.set_xlabel("")

for lab, color in [(0, "#ff6b6b"), (1, "#7ee787")]:
    ax2.hist(all_rows.loc[all_rows["label"] == lab, "length"], bins=range(7, 14),
             alpha=0.65, label="negative" if lab == 0 else "positive", color=color)
ax2.set_title("sentence lengths")
ax2.set_xlabel("tokens")
ax2.set_ylabel("count")
ax2.legend()
plt.tight_layout(); plt.show()

### Step 4 — Build the vocabulary (`word -> id`)

Neural networks consume integer ids, not strings. We reserve id `0` for `<pad>` and assign every word in train/test a stable id in first-seen order. The `kind` column lets you see the three roles: padding, neutral filler, and sentiment-bearing words.

In [ ]:
vocab = {"<pad>": 0}
for s, _ in train + test:
    for w in tok(s):
        if w not in vocab:
            vocab[w] = len(vocab)
itos = {i: w for w, i in vocab.items()}
V = len(vocab)

def word_kind(w):
    if w == "<pad>": return "padding"
    if w in pos_words: return "positive"
    if w in neg_words: return "negative"
    return "neutral"

vocab_df = pd.DataFrame({
    "word": list(vocab.keys()),
    "id": list(vocab.values()),
    "kind": [word_kind(w) for w in vocab.keys()],
})
print("vocabulary size V =", V)
display(vocab_df.sort_values("id").reset_index(drop=True))

### Step 5 — Tokenize and pad a batch

Sentences have different lengths, so a batch must be padded to a rectangle. `X` stores word ids; `mask` stores `1` for real tokens and `0` for padding. Later, attention uses the mask to give padding positions zero probability.

In [ ]:
def encode(s):
    return [vocab[w] for w in tok(s)]

def batchify(data):                       # pad to the longest sentence, build a mask
    seqs = [encode(s) for s, _ in data]
    L = max(len(x) for x in seqs)
    X = torch.zeros(len(seqs), L, dtype=torch.long)
    mask = torch.zeros(len(seqs), L)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)   # real tokens
        mask[i, :len(s)] = 1              # 1 where there's a real token, 0 for padding
    y = torch.tensor([lab for _, lab in data])
    return X, mask, y

Xtr, Mtr, ytr = batchify(train)
Xte, Mte, yte = batchify(test)

print("Xtr:", tuple(Xtr.shape), "Mtr:", tuple(Mtr.shape), "ytr:", tuple(ytr.shape))
print("Xte:", tuple(Xte.shape), "Mte:", tuple(Mte.shape), "yte:", tuple(yte.shape))

### Step 6 — Visualize padding and the mask

The heatmaps below show one small padded batch. White/blank-ish cells in the mask are padding. This is the same mask the attention layer will use to avoid treating `<pad>` as a meaningful word.

In [ ]:
demo_batch = train[:8]
demo_X, demo_M, demo_y = batchify(demo_batch)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
im1 = ax1.imshow(demo_X.numpy(), aspect="auto", cmap="Blues")
ax1.set_title("padded token ids")
ax1.set_xlabel("position"); ax1.set_ylabel("example")
fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

im2 = ax2.imshow(demo_M.numpy(), aspect="auto", cmap="Greens", vmin=0, vmax=1)
ax2.set_title("mask: 1=real, 0=pad")
ax2.set_xlabel("position"); ax2.set_ylabel("example")
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

display(pd.DataFrame({
    "example": range(len(demo_batch)),
    "label": ["positive" if y == 1 else "negative" for _, y in demo_batch],
    "sentence": [s for s, _ in demo_batch],
}))

## Reference implementation — PyTorch

The finished classifier follows a classic NLP pipeline:

| Component | Job | In our code |
|---|---|---|
| word embeddings | turn word ids into vectors | `nn.Embedding(V, EMB, padding_idx=0)` |
| recurrent encoder | read left-to-right and right-to-left context | `nn.LSTM(..., bidirectional=True)` |
| attention pooling | score tokens, mask padding, weighted-sum one vector | `nn.Linear(2*HID, 1)` + `softmax` |
| classifier | map the sentence vector to negative/positive logits | `nn.Linear(2*HID, 2)` |

We will build the attention-pooling head **tensor by tensor** on one sentence before wrapping it into `SentimentNet`.

### Step 7 — Hyperparameters and one concrete sentence

We use the original model width: 32-dimensional word embeddings and a 48-unit LSTM in each direction. The concrete sentence contains two positive words so the attention visualization has something meaningful to highlight after training.

In [ ]:
EMB, HID = 32, 48
hero_sent = "i watched the film and it was really amazing and gripping"
hero_words = tok(hero_sent)
hero_ids = torch.tensor([encode(hero_sent)])
hero_mask = torch.ones_like(hero_ids, dtype=torch.float)

print("hero sentence:", hero_sent)
print("ids shape:", tuple(hero_ids.shape), " mask shape:", tuple(hero_mask.shape))
display(pd.DataFrame({"pos": range(len(hero_words)), "word": hero_words, "id": hero_ids[0].tolist()}))

### Step 8 — Hero operation (a): embed the tokens

The embedding table is a learned lookup matrix with shape `(V, EMB)`. Looking up our sentence gives one 32-dimensional vector per token, shape `(1, L, EMB)`.

In [ ]:
torch.manual_seed(1)
hero_emb = nn.Embedding(V, EMB, padding_idx=0)
hero_E = hero_emb(hero_ids)
print("ids:", tuple(hero_ids.shape), "-> embeddings:", tuple(hero_E.shape))

plt.figure(figsize=(8, 3.2))
plt.imshow(hero_E[0].detach(), aspect="auto", cmap="coolwarm")
plt.yticks(range(len(hero_words)), hero_words)
plt.xlabel("embedding dimension")
plt.title("token embeddings")
plt.colorbar(label="value"); plt.show()

### Step 9 — Hero operation (b): run the bidirectional LSTM

A bidirectional LSTM reads the sequence forward and backward. At each token it concatenates both directions, so the per-token context width is `2 * HID = 96` and the tensor shape is `(1, L, 2H)`.

In [ ]:
torch.manual_seed(1)
hero_lstm = nn.LSTM(EMB, HID, batch_first=True, bidirectional=True)
hero_H, _ = hero_lstm(hero_E)
print("embeddings:", tuple(hero_E.shape), "-> BiLSTM contexts:", tuple(hero_H.shape))

plt.figure(figsize=(8, 3.2))
plt.imshow(hero_H[0].detach(), aspect="auto", cmap="coolwarm")
plt.yticks(range(len(hero_words)), hero_words)
plt.xlabel("context dimension (forward + backward)")
plt.title("BiLSTM context vectors")
plt.colorbar(label="value"); plt.show()

### Step 10 — Hero operation (c): score each token

Attention pooling starts with a tiny linear layer: each 96-dimensional token context becomes one raw score. Scores are not probabilities yet; they are just unnormalised preferences.

In [ ]:
torch.manual_seed(1)
hero_attn = nn.Linear(2 * HID, 1)
hero_scores = hero_attn(hero_H).squeeze(-1)
print("contexts:", tuple(hero_H.shape), "-> raw scores:", tuple(hero_scores.shape))

plt.figure(figsize=(8, 3))
plt.bar(hero_words, hero_scores[0].detach().tolist(), color="#79c0ff")
plt.xticks(rotation=35, ha="right")
plt.ylabel("raw score")
plt.title("raw attention scores")
plt.tight_layout(); plt.show()

### Step 11 — Hero operation (d): mask padding to `-1e9`

Our hero sentence has no padding, so nothing changes here. In a real padded batch, every `<pad>` score is replaced by a huge negative number; after softmax those positions receive essentially zero weight.

In [ ]:
hero_scores_masked = hero_scores.masked_fill(hero_mask == 0, -1e9)
print("mask:", hero_mask[0].tolist())
print("masked scores changed?", bool((hero_scores != hero_scores_masked).any()))

### Step 12 — Hero operation (e): softmax into attention weights

Softmax converts raw scores into nonnegative weights that sum to 1 across the sentence. Before training these weights are random-ish; after training we will repeat this plot and the sentiment words should stand out.

In [ ]:
hero_alpha = torch.softmax(hero_scores_masked, dim=1)
print("attention weights shape:", tuple(hero_alpha.shape), " sum:", round(float(hero_alpha.detach().sum()), 6))

plt.figure(figsize=(8, 3))
plt.bar(hero_words, hero_alpha[0].detach().tolist(), color="#7ee787")
plt.xticks(rotation=35, ha="right")
plt.ylabel("attention weight")
plt.title("attention weights")
plt.tight_layout(); plt.show()

**🎛️ Play with it — attention temperature**

Softmax has a sharpness knob: divide scores by temperature `T`. **Try:** low `T` like 0.2 and high `T` like 3.0. **Watch:** low temperature concentrates attention; high temperature spreads it out. **Why it matters:** attention weights are a distribution, and score scale controls how decisive that distribution becomes.

In [ ]:
fixed_words = ["the", "film", "was", "really", "amazing", "and", "gripping"]
fixed_scores = torch.tensor([-0.5, -0.2, 0.0, 0.1, 1.7, -0.1, 2.1])

def temp_play(T=1.0):
    weights = torch.softmax(fixed_scores / T, dim=0)
    plt.figure(figsize=(7.5, 3))
    plt.bar(fixed_words, weights.tolist(), color="#7ee787")
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    plt.title(f"attention temperature T={T}")
    plt.ylabel("weight")
    plt.tight_layout(); plt.show()
    print("weights sum:", round(float(weights.sum()), 6), "  top word:", fixed_words[int(weights.argmax())])

try:
    from ipywidgets import interact, FloatSlider
    interact(temp_play, T=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1))
except Exception:
    temp_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 13 — Hero operation (f): weighted sum to one sentence vector

The attention weights choose how much of each token context vector to keep. Multiplying `alpha` by the BiLSTM contexts and summing over tokens collapses `(1, L, 2H)` into one `(1, 2H)` sentence vector.

In [ ]:
hero_ctx = (hero_alpha.unsqueeze(-1) * hero_H).sum(1)
print("alpha:", tuple(hero_alpha.shape), " contexts:", tuple(hero_H.shape), "-> sentence vector:", tuple(hero_ctx.shape))

plt.figure(figsize=(8, 1.8))
plt.imshow(hero_ctx.detach(), aspect="auto", cmap="coolwarm")
plt.yticks([0], ["sentence"])
plt.xlabel("context dimension")
plt.title("one sentence vector")
plt.colorbar(label="value"); plt.show()

### Step 14 — Hero operation (g): classifier logits

The classifier is a linear layer from the sentence vector to two logits: index `0` for negative and index `1` for positive. Softmax turns those logits into probabilities.

In [ ]:
torch.manual_seed(1)
hero_cls = nn.Linear(2 * HID, 2)
hero_logits = hero_cls(hero_ctx)
hero_probs = torch.softmax(hero_logits, dim=1)
print("sentence vector:", tuple(hero_ctx.shape), "-> logits:", tuple(hero_logits.shape))
display(pd.DataFrame({"class": ["negative", "positive"],
                      "logit": hero_logits[0].detach().tolist(),
                      "probability": hero_probs[0].detach().tolist()}))

### Step 15 — Package the same operations into `SentimentNet`

The class below is exactly the hero operation wrapped into a reusable model: embed → BiLSTM → score → mask → softmax → weighted sum → classify. It returns both logits and attention weights so we can inspect the model later.

In [ ]:
class SentimentNet(nn.Module):
    def __init__(self, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        out_width = 2 * HID if bidirectional else HID
        self.emb  = nn.Embedding(V, EMB, padding_idx=0)        # word -> vector table
        self.lstm = nn.LSTM(EMB, HID, batch_first=True,
                            bidirectional=bidirectional)       # BiLSTM encoder
        self.attn = nn.Linear(out_width, 1)                    # one score per token
        self.cls  = nn.Linear(out_width, 2)                    # classifier head

    def forward(self, x, mask):
        embedded = self.emb(x)                                 # (B, L, EMB)
        h, _ = self.lstm(embedded)                             # (B, L, H) or (B, L, 2H)
        score = self.attn(h).squeeze(-1)                       # (B, L) one score per token
        score = score.masked_fill(mask == 0, -1e9)             # ignore padding positions
        alpha = torch.softmax(score, dim=1)                    # weights over tokens, sum to 1
        ctx = (alpha.unsqueeze(-1) * h).sum(1)                 # ONE sentence vector
        logits = self.cls(ctx)
        return logits, alpha                                   # logits + attention weights

net = SentimentNet()
opt = torch.optim.Adam(net.parameters(), lr=4e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss()

### Step 16 — Trace shapes through the model

A shape table is the fastest way to debug sequence models. Notice how the length dimension `L` survives through embeddings and the LSTM, then attention pooling removes it to create one vector per sentence.

In [ ]:
with torch.no_grad():
    xb = Xtr[:4]
    mb = Mtr[:4]
    embedded = net.emb(xb)
    h, _ = net.lstm(embedded)
    score = net.attn(h).squeeze(-1)
    score_masked = score.masked_fill(mb == 0, -1e9)
    alpha = torch.softmax(score_masked, dim=1)
    ctx = (alpha.unsqueeze(-1) * h).sum(1)
    logits = net.cls(ctx)

shape_df = pd.DataFrame([
    ("input ids", tuple(xb.shape)),
    ("mask", tuple(mb.shape)),
    ("embeddings", tuple(embedded.shape)),
    ("BiLSTM contexts", tuple(h.shape)),
    ("raw scores", tuple(score.shape)),
    ("attention weights", tuple(alpha.shape)),
    ("sentence vectors", tuple(ctx.shape)),
    ("logits", tuple(logits.shape)),
], columns=["stage", "shape"])
display(shape_df)

**🎛️ Play with it — move the sentiment word**

Here we use a transparent hand-made score rule: sentiment words get high scores and filler gets low scores. **Try:** choose `amazing` or `awful`, then move it through the sentence. **Watch:** the attention bar follows the sentiment word, not its position. **Why it matters:** a good attention-pooling classifier should be mostly position-invariant for this toy task.

In [ ]:
def move_word_play(sentiment="amazing", position=4):
    base = ["i", "watched", "the", "film", "and", "it", "was", "really", "quite", "today"]
    position = max(0, min(position, len(base)))
    words = base[:position] + [sentiment] + base[position:]
    scores = torch.tensor([2.5 if w in pos_words or w in neg_words else -0.4 for w in words])
    weights = torch.softmax(scores, dim=0)
    plt.figure(figsize=(8, 3))
    plt.bar(words, weights.tolist(), color=["#ff6b6b" if w in neg_words else "#7ee787" if w in pos_words else "#79c0ff" for w in words])
    plt.xticks(rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title("attention follows sentiment")
    plt.ylabel("weight")
    plt.tight_layout(); plt.show()
    print("top attention word:", words[int(weights.argmax())])

try:
    from ipywidgets import interact, Dropdown, IntSlider
    interact(move_word_play,
             sentiment=Dropdown(options=["amazing", "gripping", "awful", "boring"], value="amazing"),
             position=IntSlider(value=4, min=0, max=10, step=1))
except Exception:
    move_word_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 17 — Count parameters and check the untrained baseline

The parameter table shows where capacity lives. The baseline check asks: before training, is the model near chance? For two balanced classes, random guessing gives cross-entropy near `ln(2) ≈ 0.693` and accuracy near 50%.

In [ ]:
import re

group_counts = {}
for name, p in net.named_parameters():
    comp = name.split(".")[0]
    group_counts[comp] = group_counts.get(comp, 0) + p.numel()

total = sum(group_counts.values())
param_df = pd.DataFrame({
    "component": list(group_counts),
    "parameters": list(group_counts.values()),
    "% of total": [round(100 * v / total, 1) for v in group_counts.values()],
})
display(param_df)
print("total trainable parameters:", f"{total:,}")

with torch.no_grad():
    logits0, _ = net(Xte, Mte)
    loss0 = lossf(logits0, yte).item()
    acc0 = (logits0.argmax(1) == yte).float().mean().item()
print(f"untrained loss = {loss0:.3f} vs ln(2) = {float(np.log(2)):.3f}")
print(f"untrained accuracy = {acc0:.3f} (chance is about 0.500)")

**🎛️ Play with it — bidirectional vs forward-only shapes**

A bidirectional LSTM concatenates a forward and backward hidden state. **Try:** toggle bidirectional off. **Watch:** the context width changes from `2H` to `H`, and the attention/classifier layers shrink too. **Why it matters:** bidirectionality lets a token representation know both its left and right context.

In [ ]:
def direction_play(bidirectional=True):
    width = 2 * HID if bidirectional else HID
    tiny = SentimentNet(bidirectional=bidirectional)
    with torch.no_grad():
        e = tiny.emb(hero_ids)
        h, _ = tiny.lstm(e)
        logits, alpha = tiny(hero_ids, hero_mask)
    display(pd.DataFrame([
        ("embedding", tuple(e.shape)),
        ("LSTM output", tuple(h.shape)),
        ("attention weights", tuple(alpha.shape)),
        ("logits", tuple(logits.shape)),
    ], columns=["stage", "shape"]))
    print("context width:", width, "=", "2H" if bidirectional else "H")

try:
    from ipywidgets import interact, Checkbox
    interact(direction_play, bidirectional=Checkbox(value=True))
except Exception:
    direction_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

### Step 18 — What one training step does

Before the full loop, watch one Adam update on a single mini-batch. Measuring the same batch before and after the update should show the loss move downhill.

In [ ]:
xb, mb, yb = Xtr[:32], Mtr[:32], ytr[:32]
before = lossf(net(xb, mb)[0], yb)
opt.zero_grad(); before.backward(); opt.step()
after = lossf(net(xb, mb)[0], yb)
print(f"loss on this batch: before = {before.item():.3f} -> after one step = {after.item():.3f}")

### Step 19 — Train the classifier and record the loss curve

Now the real run. We start from a fresh model so the curve begins at initialization, then train for the original 80 epochs. The verifier's `--fast` mode temporarily shrinks this loop for a smoke test; the saved notebook still contains the full training loop. Our run reaches about **0.917 test accuracy**.

In [ ]:
torch.manual_seed(1)
net = SentimentNet()
opt = torch.optim.Adam(net.parameters(), lr=4e-3, weight_decay=1e-4)
loss_hist = []

for ep in range(80):
    perm = torch.randperm(len(ytr))                # fresh shuffle each epoch
    epoch_losses = []
    for i in range(0, len(ytr), 32):               # mini-batches of 32
        idx = perm[i:i + 32]
        logits, _ = net(Xtr[idx], Mtr[idx])
        loss = lossf(logits, ytr[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_losses.append(loss.item())
    loss_hist.append((ep + 1, float(np.mean(epoch_losses))))
    if (ep + 1) % 20 == 0:
        print(f"epoch {ep+1:2d}  train loss {loss_hist[-1][1]:.3f}")

**🎛️ Play with it — sentence length is not the shortcut**

This toy generator balances labels across the same length range. **Try:** choose a length. **Watch:** positive and negative examples at that length have similar counts; the label changes because of sentiment words, not length. **Why it matters:** always inspect whether an easy spurious feature could explain your model's accuracy.

In [ ]:
def length_play(length=9):
    subset = all_rows[all_rows["length"] == length]
    counts = subset.groupby("label name").size().reindex(["negative", "positive"], fill_value=0)
    plt.figure(figsize=(4.8, 3))
    plt.bar(counts.index, counts.values, color=["#ff6b6b", "#7ee787"])
    plt.title(f"labels at length {length}")
    plt.ylabel("count")
    plt.show()
    print("total examples at this length:", int(counts.sum()))

try:
    from ipywidgets import interact, IntSlider
    interact(length_play, length=IntSlider(value=9, min=7, max=12, step=1))
except Exception:
    length_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — mask padding in action**

Padding should not receive attention. **Try:** increase the padded length for a short sentence. **Watch:** padded positions keep weight near zero because their scores are masked to `-1e9`. **Why it matters:** without masking, attention could waste probability mass on fake tokens.

In [ ]:
def mask_play(total_len=12):
    words = ["really", "amazing"]
    ids = [vocab[w] for w in words]
    total_len = max(total_len, len(words))
    x = torch.zeros(1, total_len, dtype=torch.long)
    m = torch.zeros(1, total_len)
    x[0, :len(ids)] = torch.tensor(ids)
    m[0, :len(ids)] = 1
    raw = torch.linspace(0.0, 1.0, total_len).unsqueeze(0)  # deliberately gives later pads high raw scores
    masked = raw.masked_fill(m == 0, -1e9)
    weights = torch.softmax(masked, dim=1)[0]
    labels = words + ["<pad>"] * (total_len - len(words))
    plt.figure(figsize=(8, 3))
    plt.bar(range(total_len), weights.tolist(), color=["#7ee787" if lab != "<pad>" else "#cccccc" for lab in labels])
    plt.xticks(range(total_len), labels, rotation=35, ha="right")
    plt.ylim(0, 1)
    plt.title("masked pads get zero")
    plt.ylabel("attention weight")
    plt.tight_layout(); plt.show()
    print("pad weight total:", round(float(weights[len(words):].sum()), 8))

try:
    from ipywidgets import interact, IntSlider
    interact(mask_play, total_len=IntSlider(value=12, min=2, max=16, step=1))
except Exception:
    mask_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## Visualize the data & results

_Does the finished pipeline classify sentiment correctly, and do its attention weights actually land on the sentiment-bearing words?_ We answer with the recorded training history, test accuracy, and an attention bar chart for a hand-written positive sentence — no retraining needed.

### Step 20 — Plot the training loss curve

The loss should drop as the embedding, LSTM, attention, and classifier parameters learn the toy task. Exact values vary, so the headline number is labeled as **our run**.

In [ ]:
epochs = [e for e, _ in loss_hist]
losses = [l for _, l in loss_hist]
plt.figure(figsize=(7, 3.4))
plt.plot(epochs, losses, color="#7ee787", lw=1.8)
plt.axhline(float(np.log(2)), color="#ff6b6b", ls="--", label="ln(2) random")
plt.xlabel("epoch")
plt.ylabel("cross-entropy")
plt.title("training loss")
plt.legend(); plt.show()
print("first epoch loss:", round(losses[0], 3), " final epoch loss:", round(losses[-1], 3))

### Step 21 — Test accuracy and a small results table

We evaluate once in `eval()` mode with gradients off. The original run for this notebook reports **test accuracy: 0.917 (our run)**; reruns can differ slightly because the dataset and optimizer are intentionally small.

In [ ]:
net.eval()
with torch.no_grad():
    test_logits, test_alpha = net(Xte, Mte)
    preds = test_logits.argmax(1)
    acc = (preds == yte).float().mean().item()
print("test accuracy (this execution):", round(acc, 3))
print("test accuracy (our run claim): 0.917")

results_df = pd.DataFrame({
    "metric": ["test accuracy (this execution)", "test accuracy (our run claim)", "test errors", "test size"],
    "value": [round(acc, 3), 0.917, int((preds != yte).sum()), len(yte)],
})
display(results_df)

cm = pd.crosstab(pd.Series(yte.tolist(), name="actual"),
                 pd.Series(preds.tolist(), name="predicted"), dropna=False)
cm = cm.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
cm.index = ["negative", "positive"]
cm.columns = ["negative", "positive"]
display(cm)

### Step 22 — Attention on a hand-written sentence

Now we revisit the hero sentence **after training**. The two largest weights should land on `amazing` and `gripping` (our run), showing the model learned to read sentiment where it lives.

In [ ]:
sent = "i watched the film and it was really amazing and gripping"
x = torch.tensor([encode(sent)])
m = torch.ones_like(x, dtype=torch.float)
with torch.no_grad():
    logits, alpha = net(x, m)
    probs = torch.softmax(logits, dim=1)[0]
words = tok(sent)
weights = alpha[0].detach().tolist()
order = sorted(range(len(words)), key=lambda i: weights[i], reverse=True)

plt.figure(figsize=(8, 3.2))
colors = ["#7ee787" if w in pos_words else "#79c0ff" for w in words]
plt.bar(words, weights, color=colors)
plt.xticks(rotation=35, ha="right")
plt.ylabel("attention weight")
plt.title("trained attention")
plt.tight_layout(); plt.show()

print("sentence:", sent)
print("prediction:", "positive" if logits.argmax(1).item() == 1 else "negative",
      "  confidence:", round(float(probs.max()), 3))
print("top two attention words:", [words[i] for i in order[:2]])
# Our run: the two largest weights land on 'gripping' and 'amazing'; test accuracy: 0.917.

**🎛️ Play with it — type your own sentence and inspect attention**

After training, the model can score any sentence made from the toy vocabulary. **Try:** swap `amazing` for `boring`, or mix positive and negative words. **Watch:** the prediction probabilities and attention bars move. **Why it matters:** attention makes the classifier inspectable token by token.

In [ ]:
def predict_play(s="i watched the film and it was really amazing and gripping"):
    words = [w for w in tok(s.lower()) if w in vocab and w != "<pad>"]
    if not words:
        print("No known vocabulary words. Try words from the toy dataset.")
        return
    x = torch.tensor([[vocab[w] for w in words]])
    m = torch.ones_like(x, dtype=torch.float)
    net.eval()
    with torch.no_grad():
        logits, alpha = net(x, m)
        probs = torch.softmax(logits, dim=1)[0]
    pred = "positive" if int(logits.argmax(1)) == 1 else "negative"
    plt.figure(figsize=(8, 3))
    plt.bar(words, alpha[0].detach().tolist(), color=["#ff6b6b" if w in neg_words else "#7ee787" if w in pos_words else "#79c0ff" for w in words])
    plt.xticks(rotation=35, ha="right")
    plt.ylabel("attention weight")
    plt.title("your sentence attention")
    plt.tight_layout(); plt.show()
    display(pd.DataFrame({"class": ["negative", "positive"], "probability": probs.detach().tolist()}))
    print("prediction:", pred)

try:
    from ipywidgets import interact, Text
    interact(predict_play, s=Text(value="i watched the film and it was really amazing and gripping"))
except Exception:
    predict_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

**🎛️ Play with it — embedding nearest neighbours**

The embedding table began random, then training nudged useful words into geometry. **Try:** choose positive, negative, and neutral words. **Watch:** sentiment words often have nearby sentiment neighbours. **Why it matters:** embeddings are the bridge from discrete words to continuous meaning.

In [ ]:
def neighbours_play(word="amazing", k=6):
    if word not in vocab:
        print("pick a vocabulary word")
        return
    E = net.emb.weight.detach()
    En = E / E.norm(dim=1, keepdim=True).clamp_min(1e-9)
    sims = En @ En[vocab[word]]
    order = [j for j in sims.argsort(descending=True).tolist() if j != vocab[word]][:k]
    labels = [itos[j] for j in order]
    vals = [float(sims[j]) for j in order]
    plt.figure(figsize=(7, 3))
    plt.bar(labels, vals, color="#c89bff")
    plt.ylim(-1, 1)
    plt.ylabel("cosine")
    plt.title(f"neighbours of {word}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout(); plt.show()

try:
    from ipywidgets import interact, Dropdown, IntSlider
    interact(neighbours_play,
             word=Dropdown(options=sorted(vocab.keys()), value="amazing"),
             k=IntSlider(value=6, min=2, max=12, step=1))
except Exception:
    neighbours_play()        # no ipywidgets (or non-notebook) -> render sensible defaults

## Practice

1. Add a few new positive and negative words to the pools, rebuild the dataset, and see whether nearest-neighbour clusters change.
2. Replace attention pooling with mean pooling. Does accuracy change? Do you lose interpretability?
3. Try making the dataset harder by injecting both a positive and a negative word into some sentences. How should the label be chosen?